In [8]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Knowledge base (keep RAG and Vector DB documents intact)
documents = [
    "The Eiffel Tower is located in Paris, France and was completed in 1889.",
    "Python is a popular high-level programming language used in AI development.",
    "Retrieval-Augmented Generation combines document retrieval with text generation.",
    "Vector databases store embeddings and support fast similarity search."
]

# 2. Embed documents
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embed_model.encode(documents)

# 3. Build FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

# 4. Query and retrieve top-2 relevant chunks
query = "What is RAG in AI?"
query_embedding = embed_model.encode([query])

# Search for the 2 nearest neighbors
D, I = index.search(np.array(query_embedding), k=2)

# Ensure retrieved chunks match the exact indices returned by FAISS
retrieved_chunks = [documents[i] for i in I[0]]

# 5. Load Model & Tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

# Clear QA prompt template for FLAN-T5
context = " ".join(retrieved_chunks)
prompt = f"answer the question using the context below.\n\nContext: {context}\n\nQuestion: {query}"

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=50)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# 6. Print Results
print(f"Question: {query}")
print(f"Retrieved Context: {retrieved_chunks}")
print(f"Answer: {generated_text}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Question: What is RAG in AI?
Retrieved Context: ['Python is a popular high-level programming language used in AI development.', 'Retrieval-Augmented Generation combines document retrieval with text generation.']
Answer: combines document retrieval with text generation


In [2]:
!pip install faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 73.7 MB/s eta 0:00:00
